In [1]:
# ==========================================================
# Imports
# ==========================================================

from pathlib import Path

import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
import random
import numpy as np
import torch

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [4]:
PROJECT_PATH = Path("/content/drive/MyDrive/UAH_Project")

DATASET_PATH = (
    PROJECT_PATH
    / "datasets"
    / "processed"
    / "uah_dataset_window80.npz"
)

sensor_data = np.load(DATASET_PATH)

X = sensor_data["X"]
y = sensor_data["y"]
groups = sensor_data["groups"]

print(X.shape)

(30836, 80, 13)


In [5]:
unique_groups = np.unique(groups)

trip_labels = np.array([
    y[groups == trip][0]
    for trip in unique_groups
])

train_groups, test_groups = train_test_split(
    unique_groups,
    test_size=0.20,
    random_state=42,
    stratify=trip_labels,
)

In [6]:
train_mask = np.isin(
    groups,
    train_groups,
)

test_mask = np.isin(
    groups,
    test_groups,
)

X_train = X[train_mask]
X_test = X[test_mask]

y_train = y[train_mask]
y_test = y[test_mask]

In [7]:
print(X_train.shape)
print(X_test.shape)

(24853, 80, 13)
(5983, 80, 13)


In [8]:
from sklearn.preprocessing import StandardScaler

# (samples, timesteps, features)
n_train, seq_len, n_features = X_train.shape
n_test = X_test.shape[0]

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train.reshape(-1, n_features)
).reshape(n_train, seq_len, n_features)

X_test_scaled = scaler.transform(
    X_test.reshape(-1, n_features)
).reshape(n_test, seq_len, n_features)

print(X_train_scaled.shape)
print(X_test_scaled.shape)

(24853, 80, 13)
(5983, 80, 13)


In [9]:
from torch.utils.data import Dataset

class DriverDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [10]:
from torch.utils.data import DataLoader

batch_size = 32

train_dataset = DriverDataset(X_train_scaled, y_train)
test_dataset = DriverDataset(X_test_scaled, y_test)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [11]:
import torch.nn as nn

class LSTMClassifier(nn.Module):

    def __init__(
        self,
        input_size=13,
        hidden_size=64,
        num_layers=2,
        num_classes=3,
        dropout=0.3
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):

        output, (hidden, cell) = self.lstm(x)

        x = hidden[-1]

        x = self.dropout(x)

        x = self.fc(x)

        return x

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cpu


In [13]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(device)

print(class_weights)

tensor([0.8212, 0.9873, 1.2997])


In [14]:
model = LSTMClassifier().to(device)

In [15]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)

In [16]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4
)

In [17]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        # Gradient Clipping
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [18]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)

            loss = criterion(outputs, y_batch)

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    loss = running_loss / len(loader)

    acc = accuracy_score(all_labels, all_preds)

    macro_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    return loss, acc, macro_f1, all_labels, all_preds

In [19]:
import copy

num_epochs = 20
patience = 3

best_model = None
best_f1 = 0.0
patience_counter = 0

train_losses = []
test_losses = []

train_accs = []
test_accs = []

test_f1s = []

for epoch in range(num_epochs):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    test_loss, test_acc, test_f1, _, _ = evaluate(
        model,
        test_loader,
        criterion,
        device
    )

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    train_accs.append(train_acc)
    test_accs.append(test_acc)

    test_f1s.append(test_f1)

    print(
        f"Epoch {epoch+1:02d}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Test Loss: {test_loss:.4f} | "
        f"Test Acc: {test_acc:.4f} | "
        f"Macro F1: {test_f1:.4f}"
    )

    if test_f1 > best_f1:

        best_f1 = test_f1
        patience_counter = 0

        best_model = copy.deepcopy(model.state_dict())


    else:

        patience_counter += 1

    if patience_counter >= patience:

        print("\nEarly stopping!")
        break

Epoch 01/20 | Train Loss: 0.9201 | Train Acc: 0.5012 | Test Loss: 0.8697 | Test Acc: 0.5098 | Macro F1: 0.5186
Epoch 02/20 | Train Loss: 0.7153 | Train Acc: 0.6585 | Test Loss: 0.8068 | Test Acc: 0.5816 | Macro F1: 0.6013
Epoch 03/20 | Train Loss: 0.6365 | Train Acc: 0.7029 | Test Loss: 0.7743 | Test Acc: 0.6437 | Macro F1: 0.6465
Epoch 04/20 | Train Loss: 0.6047 | Train Acc: 0.7194 | Test Loss: 0.7440 | Test Acc: 0.6647 | Macro F1: 0.6714
Epoch 05/20 | Train Loss: 0.5738 | Train Acc: 0.7327 | Test Loss: 0.7412 | Test Acc: 0.6637 | Macro F1: 0.6681
Epoch 06/20 | Train Loss: 0.5471 | Train Acc: 0.7487 | Test Loss: 0.7751 | Test Acc: 0.6604 | Macro F1: 0.6661
Epoch 07/20 | Train Loss: 0.5311 | Train Acc: 0.7555 | Test Loss: 0.7752 | Test Acc: 0.6622 | Macro F1: 0.6669

Early stopping!


In [20]:
model.load_state_dict(best_model)

<All keys matched successfully>

In [21]:
test_loss, test_acc, test_f1, y_true, y_pred = evaluate(
    model,
    test_loader,
    criterion,
    device
)

print(f"\nFinal Accuracy : {test_acc:.4f}")
print(f"Final Macro F1 : {test_f1:.4f}")


Final Accuracy : 0.6647
Final Macro F1 : 0.6714


In [22]:
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix

print(classification_report(y_true, y_pred))

cm = confusion_matrix(y_true, y_pred)

print(cm)

              precision    recall  f1-score   support

           0       0.73      0.58      0.65      2971
           1       0.49      0.63      0.56      1503
           2       0.77      0.86      0.81      1509

    accuracy                           0.66      5983
   macro avg       0.66      0.69      0.67      5983
weighted avg       0.68      0.66      0.67      5983

[[1721  882  368]
 [ 524  953   26]
 [ 110   96 1303]]


# Experiment 2 - Dropout Analysis

In [23]:
# Dropout Experiment (0.2)

model = LSTMClassifier(
    input_size=X_train.shape[2],
    hidden_size=64,
    num_layers=2,
    num_classes=len(np.unique(y)),
    dropout=0.2
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [24]:
from sklearn.metrics import accuracy_score, f1_score

In [25]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0

    all_preds = []
    all_labels = []

    for X_batch, y_batch in loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        # Gradient Clipping
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

    epoch_loss = running_loss / len(loader)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    return epoch_loss, epoch_acc, epoch_f1

In [26]:
def evaluate(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0

    all_preds = []
    all_labels = []

    with torch.no_grad():

        for X_batch, y_batch in loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)

            loss = criterion(outputs, y_batch)

            running_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    epoch_loss = running_loss / len(loader)
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(
        all_labels,
        all_preds,
        average="macro"
    )

    return epoch_loss, epoch_acc, epoch_f1

In [27]:
num_epochs = 20

best_f1 = 0
best_model = copy.deepcopy(model.state_dict())
patience = 3
counter = 0

train_losses = []
test_losses = []

train_accs = []
test_accs = []

train_f1s = []
test_f1s = []

for epoch in range(num_epochs):

    train_loss, train_acc, train_f1 = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    test_loss, test_acc, test_f1 = evaluate(
        model,
        test_loader,
        criterion,
        device
    )

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    train_accs.append(train_acc)
    test_accs.append(test_acc)

    train_f1s.append(train_f1)
    test_f1s.append(test_f1)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
    print(f"Test  Loss: {test_loss:.4f} | Acc: {test_acc:.4f} | F1: {test_f1:.4f}")

    if test_f1 > best_f1:
        best_f1 = test_f1
        best_model = copy.deepcopy(model.state_dict())
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping!")
        break

Epoch 1/20
Train Loss: 0.9074 | Acc: 0.5229 | F1: 0.5007
Test  Loss: 0.8865 | Acc: 0.4951 | F1: 0.5102
Epoch 2/20
Train Loss: 0.6870 | Acc: 0.6676 | F1: 0.6754
Test  Loss: 0.8872 | Acc: 0.5295 | F1: 0.5429
Epoch 3/20
Train Loss: 0.6298 | Acc: 0.6953 | F1: 0.7053
Test  Loss: 0.8687 | Acc: 0.5552 | F1: 0.5751
Epoch 4/20
Train Loss: 0.5953 | Acc: 0.7136 | F1: 0.7231
Test  Loss: 0.8020 | Acc: 0.5790 | F1: 0.5973
Epoch 5/20
Train Loss: 0.5640 | Acc: 0.7273 | F1: 0.7365
Test  Loss: 0.7769 | Acc: 0.6440 | F1: 0.6533
Epoch 6/20
Train Loss: 0.5387 | Acc: 0.7416 | F1: 0.7510
Test  Loss: 0.7849 | Acc: 0.6592 | F1: 0.6653
Epoch 7/20
Train Loss: 0.5147 | Acc: 0.7552 | F1: 0.7642
Test  Loss: 0.8908 | Acc: 0.6288 | F1: 0.6403
Epoch 8/20
Train Loss: 0.4993 | Acc: 0.7612 | F1: 0.7702
Test  Loss: 0.8691 | Acc: 0.6249 | F1: 0.6422
Epoch 9/20
Train Loss: 0.4759 | Acc: 0.7769 | F1: 0.7854
Test  Loss: 0.8779 | Acc: 0.6071 | F1: 0.6222
Early stopping!


In [28]:
model.load_state_dict(best_model)

test_loss, test_acc, test_f1 = evaluate(
    model,
    test_loader,
    criterion,
    device
)

print("=" * 50)
print("Dropout = 0.2")
print(f"Final Accuracy : {test_acc:.4f}")
print(f"Final Macro F1 : {test_f1:.4f}")
print("=" * 50)

Dropout = 0.2
Final Accuracy : 0.6592
Final Macro F1 : 0.6653


In [29]:
# Dropout Experiment (0.4)

model = LSTMClassifier(
    input_size=X_train.shape[2],
    hidden_size=64,
    num_layers=2,
    num_classes=len(np.unique(y)),
    dropout=0.4
).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [30]:
num_epochs = 20

best_f1 = 0
best_model = copy.deepcopy(model.state_dict())
patience = 3
counter = 0

train_losses = []
test_losses = []

train_accs = []
test_accs = []

train_f1s = []
test_f1s = []

for epoch in range(num_epochs):

    train_loss, train_acc, train_f1 = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    test_loss, test_acc, test_f1 = evaluate(
        model,
        test_loader,
        criterion,
        device
    )

    train_losses.append(train_loss)
    test_losses.append(test_loss)

    train_accs.append(train_acc)
    test_accs.append(test_acc)

    train_f1s.append(train_f1)
    test_f1s.append(test_f1)

    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"Train Loss: {train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
    print(f"Test  Loss: {test_loss:.4f} | Acc: {test_acc:.4f} | F1: {test_f1:.4f}")

    if test_f1 > best_f1:
        best_f1 = test_f1
        best_model = copy.deepcopy(model.state_dict())
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print("Early stopping!")
        break

Epoch 1/20
Train Loss: 0.9269 | Acc: 0.4934 | F1: 0.4797
Test  Loss: 0.9146 | Acc: 0.4889 | F1: 0.5014
Epoch 2/20
Train Loss: 0.7334 | Acc: 0.6459 | F1: 0.6549
Test  Loss: 0.8282 | Acc: 0.5577 | F1: 0.5773
Epoch 3/20
Train Loss: 0.6548 | Acc: 0.6959 | F1: 0.7054
Test  Loss: 0.7191 | Acc: 0.6786 | F1: 0.6779
Epoch 4/20
Train Loss: 0.6203 | Acc: 0.7145 | F1: 0.7240
Test  Loss: 0.7592 | Acc: 0.6348 | F1: 0.6469
Epoch 5/20
Train Loss: 0.5913 | Acc: 0.7251 | F1: 0.7345
Test  Loss: 0.8009 | Acc: 0.5848 | F1: 0.6109
Epoch 6/20
Train Loss: 0.5638 | Acc: 0.7375 | F1: 0.7471
Test  Loss: 0.8514 | Acc: 0.5933 | F1: 0.6162
Early stopping!


In [31]:
model.load_state_dict(best_model)

test_loss, test_acc, test_f1 = evaluate(
    model,
    test_loader,
    criterion,
    device
)

print("=" * 50)
print("Dropout = 0.4")
print(f"Final Accuracy : {test_acc:.4f}")
print(f"Final Macro F1 : {test_f1:.4f}")
print("=" * 50)

Dropout = 0.4
Final Accuracy : 0.6786
Final Macro F1 : 0.6779
